# 7.4 Combining Data Responsibly

**Machine learning series, class 4 of 4.** A model with a good AUC isn't finished. Before anyone uses it, three questions remain:

1. **Does it work equally well for everyone?** An AUC of 0.74 is an average. A model can rank one group well and another poorly, or be right on average and wrong for a group.
2. **Is more data better?** Each extra kind of data (a Fitbit, a blood sample) adds information, but only some people have it. Combining data **shrinks** the sample, and changes **who** is in it.
3. **Can someone else check it?** A prediction model is only as trustworthy as its report. **TRIPOD+AI** is the checklist for reporting one.

This class answers all three for the models of 7.1 to 7.3.

### By the end of this notebook you'll be able to

1. Check a model's **discrimination** and **calibration** in each group (sex, age, income, race and ethnicity)
2. Ask **who** a model would flag, and which group's future cases it would miss
3. Count how combining data types **shrinks** the sample, and how it changes who is in it
4. Test whether **Fitbit steps** add anything to enrollment data, with cross-validation and the bootstrap
5. Report a model with the **TRIPOD+AI** checklist

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Does the 7.1 model work equally well across groups? What happens to the sample, and to prediction, when we add Fitbit and proteomics? |
| 🧪 **The analysis** | 7.1's model on its test set, by group: AUC, observed vs predicted risk, the share of cases in the top 10%. The people with each combination of data. Enrollment data vs enrollment data + Fitbit steps, by 5-fold cross-validation |
| 🎯 **Why** | TRIPOD+AI asks for performance in **key subgroups**, and for how **fairness** was addressed |
| ✅ **What we conclude** | Where the model is weaker, how much each data type costs in people, and whether steps add to enrollment data |
| ⚠️ **What it can't tell us** | Why a group differs (biology, access to care, how diabetes is diagnosed and recorded); or how the model would do outside *All of Us* |
| 🔗 **Connections** | **7.1** (the model), **7.3** (proteins; cross-validation and the bootstrap), **6.2** (Fitbit steps), **3.1** (describing groups) |

### Class plan (95 minutes)

| Time | Part | You do |
|---|---|---|
| 0:00 | **1. Key concepts**: fairness checks; the cost of combining data; reporting | listen, discuss |
| 0:15 | **2. Setup**: the 7.1 model, again | run the cells |
| 0:25 | **3. Does the model work for everyone?** | run the cells, discuss |
| 0:50 | **4. Combining data: who's left?** | run the cells, discuss |
| 1:05 | **5. Do Fitbit steps add anything?** | run the cells |
| 1:15 | **6. Reporting with TRIPOD+AI** | discuss |
| 1:25 | **Wrap-up**: the series | ✋ **Your turn** |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Run time** | about 10 minutes |
| 💰 **Query cost** | about **\$1**: the cohort query, and one query for everyone's Fitbit baseline steps (as in 6.2; \$0.18 in that test run), priced first with `run_if_cheap()` |
| 📋 **Before class** | run **7.1** to the end in the same workspace (it saves `ml_t2d_2yr.csv`) |

> ⚠️ **Keep All of Us data in the Workbench.** Share only summaries, with no counts of 1 to 20. Clear outputs before committing.

## Part 1. Key concepts

### Three ways a model can be unfair

| Check | Asks | A problem looks like |
|---|---|---|
| **Discrimination** in each group | Within this group, does the model rank future cases above others? | AUC 0.78 for one group, 0.65 for another |
| **Calibration** in each group | Within this group, does the average predicted risk match the share diagnosed? | predicted 5%, observed 8%: the model **under**-predicts, and the group's risk looks lower than it is |
| **Who gets flagged** | If a prevention program reached the top 10% of predicted risk, what share of each group's **future cases** would it reach? | it reaches 35% of one group's cases, but 15% of another's |

The **observed ÷ expected** ratio (O/E) sums up calibration: 1 is right, above 1 means the model **under**-predicts for that group, below 1 that it **over**-predicts.

**Why groups can differ:** some groups have fewer people to learn from; a predictor can mean different things in different groups (the same BMI carries different risk at different ages, and in different populations); and the **outcome itself** depends on care: a person who sees a doctor less often is diagnosed later, or not at all, so their "no diabetes" is less certain.

**Small groups can't be checked.** With few cases, a group's AUC is too uncertain to read. We combine groups with fewer than 50 new cases: they're not audited, which is a limitation to report, not a reassurance.

### The cost of combining data

Each data type has its own **time zero** (7.3), and only some people have it:

```
Enrollment data        ████████████████████████████████████████  everyone in 7.1
+ Fitbit               ████████                                  people who shared a Fitbit
+ proteomics           █                                         people chosen for proteomics
+ Fitbit + proteomics  ▏                                         both: very few
```

The people who are left may **differ** from everyone else: Fitbit users may be younger, wealthier, more active. A model built on them may not work for others.

### Reporting: TRIPOD+AI

**TRIPOD+AI** (Collins and others, *BMJ*, 2024) is a 27-item checklist for reporting prediction models, whether built with regression or machine learning. It asks, among other things, for performance **with confidence intervals, including for key subgroups** (item 23a), and how **fairness** was addressed (item 14). Part 6 fills it in for this series.

## Part 2. Setup: the 7.1 model, again

We read 7.1's file, make the **same split** (the same seed and the same rows give the same test set), and refit 7.1's model 3: logistic regression on all enrollment data. The cohort query adds each person's **self-reported race and ethnicity category** and their dates.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

install_if_missing(c("rsample", "broom"))
suppressPackageStartupMessages({library(rsample); library(broom)})

HORIZON_YEARS <- 2
end_of_data   <- as.Date("2025-01-01")
PREDICTORS    <- c("age", "sex_at_birth", "bmi", "sbp", "general_health", "income", "education", "smoking_status")

cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id),
         t2d_date  = as.Date(first_t2d_date),
         last_ehr  = as.Date(last_ehr_date),
         death     = as.Date(death_date))

ml_data <- load_data(str_glue("ml_t2d_{HORIZON_YEARS}yr.csv"), col_types = cols(person_id = "c")) %>%
  mutate(income         = str_remove_all(income, "\\$"),          # files saved before November 2026 had $ signs
         t2d_future     = factor(t2d_future, levels = c("no", "yes")),
         sex_at_birth   = factor(sex_at_birth),
         general_health = factor(general_health, levels = c("Excellent", "Very Good", "Good", "Fair", "Poor", "Missing")),
         income         = factor(income, levels = c("100k or more", "50k to 100k", "25k to 50k", "Under 25k",
                                                    "Prefer not to answer", "Missing")),
         education      = factor(education, levels = c("College degree", "Some college", "High school or GED",
                                                       "Less than high school", "Missing")),
         smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current", "Missing")))

set.seed(2026)
split <- initial_split(ml_data, prop = 0.75, strata = t2d_future)   # the same rows as 7.1's split
train <- training(split)
test  <- testing(split)

bind_rows(training = count(train, t2d_future), test = count(test, t2d_future), .id = "set")

**🔍 Read the output:** the training and test counts: the same as 7.1's (108,030 and 6,409; 36,069 and 2,078 in our test run). If not, 7.1's file has changed: rerun 7.1.

`glm()` fits the same logistic regression as 7.1's tidymodels workflow (which uses `glm()` inside), so the test AUC should match 7.1's model 3.

In [ ]:
model_formula <- reformulate(PREDICTORS, response = "t2d_future")
fit <- glm(model_formula, data = train, family = binomial)

# From 7.3: the AUC from ranks, fast enough to bootstrap
auc <- function(y, risk) {
  r  <- rank(risk)
  n1 <- sum(y == 1); n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
}

NON_ANSWERS <- c("PMI: Skip", "PMI: Prefer Not To Answer", "I prefer not to answer", "Prefer not to answer",
                 "None Indicated", "None", "No matching concept", "PMI: Dont Know", "I don't know")   # from 2.1

scored <- test %>%
  mutate(risk = predict(fit, newdata = test, type = "response"),
         y    = as.integer(t2d_future == "yes"),
         flag = risk >= quantile(risk, 0.90)) %>%                     # the top 10% of predicted risk
  left_join(cohort %>% select(person_id, self_reported_category), by = "person_id") %>%
  mutate(race_ethnicity = if_else(is.na(self_reported_category) | self_reported_category %in% NON_ANSWERS,
                                  "Not reported", self_reported_category),
         age_group      = cut(age, breaks = c(18, 40, 60, 75, Inf), right = FALSE,
                              labels = c("18 to 39", "40 to 59", "60 to 74", "75 and older")))

round(auc(scored$y, scored$risk), 3)

**🔍 Read the output:** the test AUC: 0.743 in our test run, as in 7.1 and 7.2.

`flag` marks the **top 10%** of predicted risk: the people a prevention program might contact first. In 7.1, they included about 30% of all future cases.

## Part 3. Does the model work for everyone?

### The audit function

`audit()` takes one grouping (sex, age group, income, or race and ethnicity) and, for each group, computes:

| Column | Means |
|---|---|
| `people`, `new_t2d` | people in the test set, and how many were diagnosed within 2 years |
| `predicted_pct`, `observed_pct` | the average predicted risk, and the share actually diagnosed |
| `o_to_e` | observed ÷ predicted: 1 is right; above 1, the model **under**-predicts for this group |
| `auc`, `auc_low`, `auc_high` | the AUC **within** the group, with a 95% bootstrap confidence interval |
| `pct_flagged` | the share of the group in the top 10% |
| `pct_cases_reached` | the share of the group's **future cases** who were in the top 10% |

**Protecting small counts.** Groups with fewer than 50 new cases are combined. Counts of 1 to 20 are hidden, and so are counts that could be worked out from others: the groups add up to the whole test set, so if the hidden counts add up to 1 to 20, the next smallest is hidden too (7.1). The percentages and AUCs of hidden counts are hidden with them.

In [ ]:
# From 7.1 and 7.3, with one addition: `also` marks counts that must be hidden for another reason
hide_with_total <- function(n, also = FALSE) {
  hidden <- n <= 20 | also
  while (any(hidden) && !all(hidden) && sum(n[hidden]) %in% 1:20) {
    hidden[which.min(replace(n, hidden, Inf))] <- TRUE
  }
  replace(n, hidden, NA)
}

MIN_CASES <- 50   # groups with fewer new cases are combined: their AUC would be too uncertain

auc_ci <- function(y, risk, reps = 200) {
  boots <- replicate(reps, { i <- sample(length(y), replace = TRUE); auc(y[i], risk[i]) })
  tibble(auc = auc(y, risk), auc_low = quantile(boots, 0.025), auc_high = quantile(boots, 0.975))
}

audit <- function(data, grouping) {
  set.seed(2026)
  data %>%
    mutate(group = as.character(.data[[grouping]]),
           group = if_else(ave(y, group, FUN = sum) < MIN_CASES, "Smaller groups, combined", group)) %>%   # ave(): each group's total
    group_by(group) %>%
    summarise(people        = n(),
              new_t2d       = sum(y),
              predicted_pct = 100 * mean(risk),
              flagged       = sum(flag),
              cases_reached = sum(flag & y == 1),
              auc_ci(y, risk),                      # adds three columns: auc, auc_low, auc_high
              .groups = "drop") %>%
    mutate(people        = hide_with_total(people),
           new_t2d       = hide_with_total(new_t2d),
           flagged       = hide_with_total(flagged),
           cases_reached = hide_with_total(cases_reached,
                                           also = is.na(new_t2d) | (new_t2d - cases_reached) %in% 1:20),
           observed_pct      = 100 * new_t2d / people,
           o_to_e            = observed_pct / predicted_pct,
           pct_flagged       = 100 * flagged / people,
           pct_cases_reached = 100 * cases_reached / new_t2d,
           across(c(auc, auc_low, auc_high), ~ if_else(is.na(new_t2d) | is.na(people), NA_real_, .x)),
           grouping = grouping) %>%
    transmute(grouping, group, people, new_t2d,
              predicted_pct = round(predicted_pct, 2), observed_pct = round(observed_pct, 2),
              o_to_e = round(o_to_e, 2), auc = round(auc, 3), auc_low = round(auc_low, 3), auc_high = round(auc_high, 3),
              pct_flagged = round(pct_flagged, 1), pct_cases_reached = round(pct_cases_reached, 1))
}

### Sex at birth and age

In [ ]:
audit(scored, "sex_at_birth")
audit(scored, "age_group")

**🔍 Read the output:** one row per group.

- **AUC within a group** is usually **lower** than the overall AUC, especially for **age groups**: part of the overall AUC comes from ranking older people above younger ones, and within an age group that help is gone. That's expected, not unfair.
- **`o_to_e`** close to 1 in every group means the risks are right for everyone.
- **`pct_flagged`** will differ a lot by age: older people have higher risk. Is that a problem? Compare **`pct_cases_reached`**: the share of each group's **future cases** the program would reach.

### Income, and race and ethnicity

Race and ethnicity use *All of Us*'s **self-reported category**, from the cohort query. Non-answers are grouped as "Not reported" (as in 2.1).

In [ ]:
audit(scored, "income")
audit(scored, "race_ethnicity")

**🔍 Read the output:**

- **`o_to_e` above 1** for a group means the model **under**-predicts: the group's real risk is higher than the model says. Look at the lower income groups: income is **in** the model, so the average should be about right. For race and ethnicity, which are **not** in the model, a gap is more likely.
- **`pct_cases_reached`**: if one group's future cases are reached much less often, a program based on this model would **miss** them more.
- **Wide AUC intervals** mean too few cases to tell. Don't over-read a difference whose intervals overlap.
- **"Smaller groups, combined"** (if it appears) is not a group of people with anything in common: it's what we **couldn't audit**. Report it as a limitation.

### All groups in one picture

In [ ]:
audits <- map(c("sex_at_birth", "age_group", "income", "race_ethnicity"), ~ audit(scored, .x)) %>% list_rbind()

audits %>%
  filter(!is.na(auc)) %>%
  ggplot(aes(x = auc, y = group, xmin = auc_low, xmax = auc_high)) +
  geom_vline(xintercept = auc(scored$y, scored$risk), linetype = "dashed", colour = "grey50") +
  geom_pointrange(colour = "#3182bd") +
  facet_grid(grouping ~ ., scales = "free_y", space = "free_y",
             labeller = as_labeller(c(sex_at_birth = "Sex at birth", age_group = "Age", income = "Income",
                                      race_ethnicity = "Race and ethnicity"))) +
  labs(x = "AUC within the group (95% CI)", y = NULL,
       title = "Does the 7.1 model rank people well in every group?",
       subtitle = "Dashed line: the AUC for everyone") +
  theme_minimal(base_size = 13) +
  theme(strip.text.y = element_text(angle = 0))

**🔍 Read the output:** one point per group, with its 95% confidence interval; the dashed line is the overall AUC. Groups whose intervals sit clearly to the **left** are where the model ranks people less well.

**What could we do about a gap?** TRIPOD+AI (item 12f) asks authors to report any **recalibration** for particular groups. Options include recalibrating for a group, adding predictors that matter for it, or collecting more data from it. Whether to put **race and ethnicity themselves** into a model is debated: they're social categories, not biology, and using them can either correct a bias or build one in. Discuss below.

## Part 4. Combining data: who's left?

### Fitbit baseline steps

The same baseline as **6.2**: each person's first 6 months of **valid** Fitbit days (worn at least 10 hours, 100 to 45,000 steps), with at least 10 such days, and the median steps a day. `run_if_cheap()` prices the query first.

In [ ]:
fitbit_sql <- str_glue("
WITH wear AS (
  SELECT person_id, date, SUM(minute_in_zone) AS wear_minutes
  FROM `{CURRENT_DATASET}.heart_rate_summary`
  GROUP BY person_id, date
),
valid AS (
  SELECT a.person_id, a.date, a.steps
  FROM `{CURRENT_DATASET}.activity_summary` AS a
  JOIN wear AS w ON a.person_id = w.person_id AND a.date = w.date
  WHERE w.wear_minutes >= 600 AND a.steps BETWEEN 100 AND 45000
),
first_day AS (
  SELECT person_id, MIN(date) AS first_valid FROM valid GROUP BY person_id
)
SELECT v.person_id,
       MAX(v.date) AS baseline_end,
       COUNT(*)    AS baseline_days,
       APPROX_QUANTILES(v.steps, 100)[OFFSET(50)] AS baseline_steps
FROM valid AS v
JOIN first_day AS f ON v.person_id = f.person_id
WHERE v.date < DATE_ADD(f.first_valid, INTERVAL 6 MONTH)
GROUP BY v.person_id
HAVING COUNT(*) >= 10")

fitbit <- run_if_cheap(fitbit_sql, max_dollars = 3) %>%
  mutate(person_id = as.character(person_id), baseline_end = as.Date(baseline_end))

last_query_cost()
nrow(fitbit)   # people with a Fitbit baseline (everyone in All of Us, not only 7.1's people)

**🔍 Read the output:** the cost (cents), and how many people in *All of Us* have a Fitbit baseline.

### Proteomics and blood-draw dates

As in 7.3: the manifest says who has proteomics; the genome QC file gives the blood-draw date.

In [ ]:
PROT_DIR <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
QC_FILE  <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "aux", "qc", "genomics_metrics.tsv")

manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

draws <- read_tsv(QC_FILE, col_types = cols(.default = "c"), col_select = c(research_id, biosample_collection_date)) %>%
  transmute(person_id = research_id,
            draw_date = as.Date(parse_date_time(biosample_collection_date,
                                                orders = c("ymd HMS", "ymd HM", "ymd", "mdy HMS", "mdy HM", "mdy"),
                                                quiet = TRUE))) %>%
  filter(!is.na(draw_date)) %>%
  group_by(person_id) %>%
  summarise(draw_date = min(draw_date), .groups = "drop")

### Who's left with each combination?

`eligible()` applies the rules of 7.1 and 7.3 for a given **time zero**: no type 2 diabetes by time zero, and 2 years of follow-up after it (or a diagnosis sooner). Each combination's time zero is the **latest** of its data dates: we can only predict once we have **all** the data.

`describe()` summarises the people who are left. Percentages that would reveal a count of 1 to 20 (of the group, or of the rest of it) are hidden.

In [ ]:
base <- ml_data %>%
  left_join(cohort %>% select(person_id, cohort, t2d_date, last_ehr, death), by = "person_id") %>%
  left_join(fitbit, by = "person_id") %>%
  left_join(draws, by = "person_id") %>%
  mutate(has_fitbit     = !is.na(baseline_end),
         has_proteomics = person_id %in% manifest$research_id,
         followup_end   = pmin(last_ehr, death, end_of_data, na.rm = TRUE))

eligible <- function(data) {
  data %>%
    mutate(horizon_end = time_zero + round(365.25 * HORIZON_YEARS),
           t2d_by_zero = cohort == "case" & t2d_date <= time_zero,
           new         = cohort == "case" & t2d_date > time_zero & t2d_date <= horizon_end,
           age_at_zero = age + as.numeric(time_zero - enroll_date) / 365.25) %>%
    filter(!t2d_by_zero, new | followup_end >= horizon_end)
}

safe_pct <- function(x) {                 # a percentage that can't reveal a count of 1 to 20
  k <- sum(x); n <- length(x)
  if (k %in% 1:20 || (n - k) %in% 1:20) NA_real_ else round(100 * k / n, 1)
}

describe <- function(data, label) {
  data %>%
    summarise(data = label, people = n(), new_t2d = sum(new), pct_new_t2d = round(100 * mean(new), 2),
              pct_female = safe_pct(sex_at_birth == "Female"),
              pct_age_60_plus = safe_pct(age_at_zero >= 60),
              pct_income_under_25k = safe_pct(income == "Under 25k"),
              pct_bmi_30_plus = safe_pct(bmi >= 30))
}

combos <- list(
  "Enrollment data (7.1)"        = base %>% mutate(time_zero = enroll_date),
  "+ Fitbit"                     = base %>% filter(has_fitbit) %>% mutate(time_zero = pmax(enroll_date, baseline_end)),
  "+ proteomics (7.3)"           = base %>% filter(has_proteomics) %>% mutate(time_zero = pmax(enroll_date, draw_date, na.rm = TRUE)),
  "+ Fitbit + proteomics"        = base %>% filter(has_fitbit, has_proteomics) %>%
                                     mutate(time_zero = pmax(enroll_date, baseline_end, draw_date, na.rm = TRUE)))

who <- imap(combos, ~ describe(eligible(.x), .y)) %>% list_rbind()

# Hide counts of 1 to 20; and the last row if it differs from rows 2 or 3 by 1 to 20 (a derived count)
for (col in c("people", "new_t2d")) {
  x <- who[[col]]
  small <- x %in% 1:20 | (seq_along(x) == 4 & (abs(x[4] - x[2]) %in% 1:20 | abs(x[4] - x[3]) %in% 1:20))
  who[[col]][small] <- NA
}
who <- who %>% mutate(pct_new_t2d = if_else(is.na(new_t2d) | is.na(people), NA_real_, pct_new_t2d))
who

**🔍 Read the output:** one row per combination.

- **Row 1** should match 7.1 (152,586 people and 8,487 new cases in our test run), and **row 3** should match 7.3 (3,886 and 150). The same rules, the same people.
- **People and new cases fall** with each data type. With Fitbit **and** proteomics, there may be too few new cases to build or test a model at all.
- **The columns on the right** show **who** is left: if Fitbit users are younger, have higher incomes, or a lower BMI than everyone in row 1, a model built on them describes a **different population**.

## Part 5. Do Fitbit steps add anything?

For the people in row 2, we compare two logistic regressions, by **5-fold cross-validation** (no tuning, so one loop is enough):

- **Enrollment data:** the 8 predictors of 7.1 (age is now age at time zero)
- **Enrollment data + steps:** the same, plus median daily steps (per 1,000)

As in 7.3, every person is predicted by a model that never saw them, and the bootstrap gives a confidence interval for the **gain**.

In [ ]:
fitbit_people <- eligible(combos[["+ Fitbit"]]) %>%
  mutate(age = age_at_zero, steps_1000 = baseline_steps / 1000, y = as.integer(new),
         across(where(is.factor), fct_drop))

set.seed(2026)
assign_folds <- function(y, k) {                       # from 7.3: cases and non-cases spread evenly
  fold <- integer(length(y))
  fold[y == 1] <- sample(rep(1:k, length.out = sum(y == 1)))
  fold[y == 0] <- sample(rep(1:k, length.out = sum(y == 0)))
  fold
}
fold <- assign_folds(fitbit_people$y, 5)

cv_risk <- function(formula) {                         # out-of-fold predicted risks
  risk <- numeric(nrow(fitbit_people))
  for (k in 1:5) {
    fit_k <- glm(formula, data = fitbit_people[fold != k, ], family = binomial)
    risk[fold == k] <- predict(fit_k, newdata = fitbit_people[fold == k, ], type = "response")
  }
  risk
}

enroll_formula <- reformulate(PREDICTORS, response = "y")
steps_formula  <- reformulate(c(PREDICTORS, "steps_1000"), response = "y")

y           <- fitbit_people$y
risk_enroll <- cv_risk(enroll_formula)
risk_steps  <- cv_risk(steps_formula)

boot_gain <- replicate(1000, { i <- sample(length(y), replace = TRUE); auc(y[i], risk_steps[i]) - auc(y[i], risk_enroll[i]) })

tibble(people = length(y), new_t2d = sum(y),
       auc_enrollment = round(auc(y, risk_enroll), 3),
       auc_plus_steps = round(auc(y, risk_steps), 3),
       gain = round(auc(y, risk_steps) - auc(y, risk_enroll), 3),
       ci_low = round(quantile(boot_gain, 0.025), 3), ci_high = round(quantile(boot_gain, 0.975), 3))

**🔍 Read the output:** the people with Fitbit, their new cases, the cross-validated AUC of each model, and the gain from adding steps with its 95% confidence interval.

- Compare with **7.3**, where proteins added about 0.06. Steps are cheap and measured every day; do they add as much?
- 6.2 found that more steps go with **less** new diabetes. A strong **association** doesn't guarantee better **prediction**: if steps mostly carry information that BMI, age, and general health already carry, the gain is small.

In [ ]:
glm(steps_formula, data = fitbit_people, family = binomial) %>%
  tidy(exponentiate = TRUE, conf.int = TRUE) %>%
  filter(term %in% c("steps_1000", "bmi", "age")) %>%
  transmute(term, odds_ratio = round(estimate, 3), ci_low = round(conf.low, 3), ci_high = round(conf.high, 3))

**🔍 Read the output:** odds ratios from the model with steps, fitted on everyone in row 2: per **1,000 steps a day**, per BMI unit, and per year of age, each with the others held constant. An odds ratio below 1 for steps means more steps, lower risk, at the same BMI, age, and general health.

## Part 6. Reporting with TRIPOD+AI

**TRIPOD+AI** has 27 items. Here are the ones this series speaks to, and where. Item numbers are from the published checklist (Collins and others, *BMJ* 2024;385:e078378).

| Item | Asks for | In this series |
|---|---|---|
| 3c | known health inequalities between groups | Part 3; and the precision medicine page |
| 5a, 5b | the data source, its representativeness, and the dates | *All of Us* CDR v9; enrollment years (7.1); proteomics selection unknown (7.3) |
| 6b | eligibility criteria | the flow tables (7.1, 7.3) |
| 8a | the outcome, its time horizon, and how it's assessed | a first type 2 diabetes code within 2 years of time zero (7.1) |
| 9b | each predictor: how and **when** it was measured | at enrollment (7.1); the blood draw (7.3); Fitbit's first 6 months (7.4) |
| 10 | why the sample size was enough | the data check; events per predictor (7.3) |
| 11 | missing data | a "Missing" group for survey answers; people without BMI or blood pressure excluded (7.1) |
| 12a, 12c | how the data were split, the model, and its tuning | a 75/25 split and cross-validation (7.1, 7.2); nested cross-validation (7.3) |
| 12e | the performance measures | AUC, Brier score, calibration (7.1, 7.2) |
| 14 | how **fairness** was addressed | Part 3 |
| 18e, 18f | data and code sharing | data: through the *All of Us* Researcher Workbench; code: this GitHub repository |
| 20a | the flow of participants, with and without the outcome | the flow tables |
| 23a | performance **with confidence intervals, including for key subgroups** | 7.3 (bootstrap); Part 3 here |
| 26 | limitations: non-representative sample, sample size, overfitting, missing data | each class's "What it can't tell us" |

**What we can't report yet:**

- **External validation:** every number comes from *All of Us*. Would the model work in a health system's records?
- **Model updating** (items 12f, 24): we checked calibration by group, but didn't recalibrate.
- **Patient and public involvement** (item 19): no participants helped design these models. *All of Us* has participant partners: a real study should involve them.

## Your turn

1. **Another grouping.** Run `audit(scored, "education")`, or `audit(scored, "general_health")`. Which group has the lowest AUC? Is its interval wide?
2. **A different program.** Change the flag to the top **20%** (`quantile(risk, 0.80)`) in Part 2, and rerun Part 3. Does the gap in `pct_cases_reached` between groups shrink or grow?
3. **Steps as a curve.** In Part 5, replace `"steps_1000"` with `"splines::ns(steps_1000, df = 3)"`. Does the gain change?

## 💬 Discuss

1. A group's `o_to_e` is 1.3: the model **under**-predicts its risk by 30%. Who is harmed, and how? What would you do?
2. Should race and ethnicity be **predictors** in a diabetes risk model? What could go wrong either way?
3. Fitbit adds a little information but leaves out most people. Would you build a "Fitbit model" for a clinic? For whom?
4. Look back at the series: which choice (time zero, the outcome window, the split, the method, the data types) mattered **most** for the results?

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| The split counts differ from 7.1's | 7.1's file changed (for example, a 3-year window) | rerun 7.1 with `HORIZON_YEARS <- 2` |
| *Can't find the All of Us v9 data folder* | the data bucket isn't mounted in this app | see the setup notes in `aou_setup.R` |
| *factor has new levels* in Part 5 | a rare group appears in a test fold but not its training folds | combine rare groups with `fct_lump_min()`, or rerun: `fct_drop()` usually prevents it |
| An audit row shows `NA` everywhere | the group was hidden: too few people or new cases | expected; combine groups if you need them |
| Row 1 or row 3 in Part 4 doesn't match 7.1 or 7.3 | the files or rules differ between notebooks | check `HORIZON_YEARS`, and that 7.1's file is current |

## 🧑‍🏫 Instructor notes

- **Start with the overall AUC, then split it.** Ask students to guess which group the model works worst for, before running Part 3.
- **AUC within age groups is lower: explain why.** Age helps rank across age groups; within one, it can't. That's a property of the AUC, not unfairness. Calibration and `pct_cases_reached` are better fairness signals here.
- **"Smaller groups, combined" is a finding**, not a footnote: some groups can't be audited with these numbers. In TRIPOD+AI terms, it's a limitation (item 26).
- **Race and ethnicity** are social categories. Differences in calibration can reflect access to care and diagnosis, not biology. Keep the discussion on what the model **does**, and to whom.
- **Close the series** with the last discussion question: in our test runs, time zero, the data type (proteins added 0.06), and the people included mattered more than the method (boosting added 0.008).

## Key takeaways

- An overall AUC can hide **weaker performance in a group**: check discrimination, calibration, and who gets flagged, **by group**.
- **Small groups can't be audited** reliably: say so.
- Combining data types **shrinks** the sample and changes **who** is in it.
- A predictor can be strongly **associated** with the outcome and still add little to **prediction**.
- Report models with **TRIPOD+AI**: confidence intervals, subgroups, fairness, limitations, and code.

## The series, in one table

| Class | The lesson |
|---|---|
| **7.1** | Time zero and leakage come first; a simple logistic regression is a strong baseline |
| **7.2** | Flexible models found real curves, but added little (0.008 AUC) |
| **7.3** | New **data** (proteins) added more (0.06), with an unstable panel and a small sample |
| **7.4** | Check every group, count who's left, and report it all |

## Clean up

**Clear outputs** before committing or sharing: **Edit → Clear Outputs of All Cells**, then save. This notebook saves no files.